# 01 — Extraction

**Project:** Identifying high-risk road conditions and geographic hotspots to reduce traffic accident severity across the United States.

**Goal of this notebook:** Load the raw US Accidents dataset, inspect its structure, and confirm it meets the Capstone 2 dataset requirements (≥ 10,000 rows, ≥ 8 analytical columns, realistic data-quality issues).

**Source:** [US Accidents (Feb 2016 – Mar 2023)](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) — Moosavi, S. et al. Sourced via live traffic APIs across the contiguous United States.

**Before running:** execute `python data/raw/combine.py` once to reassemble the full CSV from the two split parts.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import normalize_columns

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'US_Accidents_March23.csv'
print('Raw file exists:', RAW_PATH.exists())
print('Raw file path  :', RAW_PATH)

## Load the raw dataset

The full dataset is ~2.9 GB / ~7.7M rows. Load with `low_memory=False` to keep dtype inference consistent across chunks.

In [ ]:
df_raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f'Rows   : {len(df_raw):,}')
print(f'Columns: {df_raw.shape[1]}')
print(f'Memory : {df_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB')

In [ ]:
df_raw.head()

In [ ]:
df_raw.info(verbose=True, show_counts=True)

## Column inventory

Normalize column names to snake_case so downstream notebooks can rely on stable identifiers.

In [ ]:
df = normalize_columns(df_raw)
print('Normalized column names:')
for c in df.columns:
    print(f'  - {c}')

## Missing-value audit

Which columns have missing data? Percentages drive the imputation strategy in notebook 02.

In [ ]:
missing = pd.DataFrame({
    'null_count': df.isna().sum(),
    'null_pct'  : (df.isna().mean() * 100).round(2),
    'dtype'     : df.dtypes.astype(str),
})
missing = missing.sort_values('null_pct', ascending=False)
missing

## Key descriptive statistics

In [ ]:
df.describe(include='all', ).T

In [ ]:
# Time range and severity distribution — the two headline dimensions of the study.
start_time = pd.to_datetime(df['start_time'], errors='coerce')
print('Date range :', start_time.min(), '→', start_time.max())
print('Unique states:', df['state'].nunique())
print('Unique cities:', df['city'].nunique())
print('\nSeverity distribution (%):')
print((df['severity'].value_counts(normalize=True).sort_index() * 100).round(2))

## Capstone 2 dataset compliance check

| Requirement | Target | Actual |
|---|---|---|
| Row count | ≥ 10,000 | see below |
| Column count | ≥ 8 meaningful | see below |
| Realistic data-quality issues | missing values / inconsistent formats | see above |
| Format | tabular, row-level | ✓ |

In [ ]:
assert len(df) >= 10_000, 'Dataset does not meet the minimum row requirement.'
assert df.shape[1] >= 8, 'Dataset does not meet the minimum column requirement.'
print('✓ Dataset meets Capstone 2 minimum size requirements.')
print(f'  Rows   : {len(df):,}')
print(f'  Columns: {df.shape[1]}')

### Takeaways for cleaning (notebook 02)

- High-null columns like `end_lat`, `end_lng`, `number`, `precipitation_in`, `wind_chill_f` will be dropped or median-imputed.
- `start_time` / `end_time` require datetime parsing to derive `duration_min`, `hour`, `day_of_week`, `month`, `season`.
- `severity` is already integer 1–4 → map to readable labels and a binary high-severity flag.
- Boolean POI columns (traffic_signal, junction, crossing, …) need explicit casting to `bool`.
- Outliers exist in `distance_mi`; we will cap at the 99th percentile.

Proceed to **`02_cleaning.ipynb`**.

# 01 Extraction

Use this notebook to load the raw dataset, inspect its structure, and confirm that the source is suitable for the project.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [ ]:
RAW_PATH = PROJECT_ROOT / 'data/raw/US_Accidents_March23.csv'
df = pd.read_csv(RAW_PATH)
df.head()

In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
df.info()